# 1) Title and objective

## OH Leuven Attendance Prediction

This notebook is the single project entry point for academic submission. It runs the existing production pipeline from `src/` and reports only the final `xgboost_log` model.

No manual feature engineering or manual training logic is implemented in this notebook.


## 2) Imports


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_DATA_DIR
from src.train import run_pipeline
from src.predict import predict_new_matches

pd.set_option("display.max_columns", None)


## 3) Load data path


In [2]:
DATA_DIR = Path(DEFAULT_DATA_DIR)
print(f"Data directory: {DATA_DIR}")


Data directory: C:\Users\ASUS\Desktop\International Project\Data


## 4) Run full pipeline (reuse existing code)


In [3]:
result = run_pipeline(
    data_dir=DATA_DIR,
    use_weather_api=False,
    use_external_transfermarkt=False,
)
print("Pipeline run completed.")


Pipeline run completed.


## 5) Extract best model and predictions


In [4]:
TARGET_MODEL = "xgboost_log"

run_info = result["run_info"]
predictions_df = result["predictions"].copy()

if TARGET_MODEL not in run_info.get("metrics_by_model", {}):
    raise ValueError(f"{TARGET_MODEL} not found in pipeline output metrics.")

pred_col = f"pred_{TARGET_MODEL}"
if pred_col not in predictions_df.columns:
    raise ValueError(f"Prediction column missing: {pred_col}")

y_test = predictions_df["actual"].to_numpy(dtype=float)
y_pred = predictions_df[pred_col].to_numpy(dtype=float)

print(f"Pipeline-selected best model: {run_info.get('best_model_name')}")
print(f"Notebook reporting model: {TARGET_MODEL}")


Pipeline-selected best model: xgboost_log
Notebook reporting model: xgboost_log


## 6) Show metrics (MAE, RMSE, MAPE, R2)


In [5]:
metrics = run_info["metrics_by_model"][TARGET_MODEL]
metrics_df = pd.DataFrame(
    [
        {
            "Model": TARGET_MODEL,
            "MAE": metrics["mae"],
            "RMSE": metrics["rmse"],
            "MAPE": metrics["mape"],
            "R2": metrics["r2"],
        }
    ]
)
display(metrics_df)


,Model,MAE,RMSE,MAPE,R2
0,xgboost_log,691.775951,946.568944,13.379518,0.157807


## 7) Plot actual vs predicted


In [6]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.8)
line_min = float(min(y_test.min(), y_pred.min()))
line_max = float(max(y_test.max(), y_pred.max()))
plt.plot([line_min, line_max], [line_min, line_max], color="red", linewidth=1.5)
plt.xlabel("Actual attendance")
plt.ylabel("Predicted attendance")
plt.title(f"Actual vs Predicted ({TARGET_MODEL})")
plt.tight_layout()
plt.show()


C:\Users\ASUS\AppData\Local\Temp\ipykernel_11548\53244324.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8) Feature importance (top 10)


In [7]:
feature_importance_df = result["feature_importance"].copy().head(10)
display(feature_importance_df)

plot_df = feature_importance_df.sort_values("importance", ascending=True)
plt.figure(figsize=(8, 5))
plt.barh(plot_df["feature"], plot_df["importance"])
plt.xlabel("Importance")
plt.title(f"Top 10 feature importance ({TARGET_MODEL})")
plt.tight_layout()
plt.show()


,feature,importance
0,numeric__attendance_last_match,0.146865
1,categorical__stage_Conference League Play-off ...,0.112329
2,categorical__opponent_tier_high,0.108925
3,categorical__away_team_Gent,0.078083
4,categorical__season_2024/2025,0.073144
5,numeric__opponent_frequency_seen,0.063281
6,numeric__attendance_last_3_avg,0.060729
7,categorical__away_team_Club Brugge,0.047186
8,numeric__month,0.041978
9,numeric__opponent_strength_score,0.030037


C:\Users\ASUS\AppData\Local\Temp\ipykernel_11548\4284180948.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9) Future prediction example (existing prediction pipeline)


In [8]:
new_matches = pd.DataFrame(
    [
        {
            "match_date": "2026-09-12",
            "away_team": "Club Brugge",
            "stage": "Regular Season",
            "kickoff_time": "20:45:00",
        }
    ]
)

future_predictions_df, future_summary = predict_new_matches(
    new_matches_df=new_matches,
    data_dir=DATA_DIR,
    use_weather_api=False,
)

display(future_predictions_df)
display(pd.DataFrame([future_summary]))


,match_date,away_team,stage,kickoff_time,raw_predicted_attendance,predicted_attendance,weather_temp_mean_c,weather_precipitation_mm,weather_rain_mm,weather_windspeed_max_kmh,weather_bad_flag,weather_source
0,2026-09-12,Club Brugge,Regular Season,20:45:00,7174.690037,7174.690037,NaN,NaN,NaN,NaN,0.0,


,best_model,rows_scored,prediction_min,prediction_mean,prediction_max,output_file,auto_generated_features,fallback_global_mean,fallback_counts,weather_api_enabled,weather_stats,weather_fallback_counts
0,xgboost_log,1,7174.690037,7174.690037,7174.690037,C:\Users\ASUS\PycharmProjects\OHL-Ai-project\o...,"[season, matchday, weekday_name, is_weekend, i...",6861.577465,"{'attendance_last_match': 0, 'attendance_last_...",False,"{'enabled': False, 'rows_requested': 1, 'rows_...","{'weather_temp_mean_c': 1, 'weather_precipitat..."


## 10) Short conclusion

This final notebook keeps the workflow short, readable, and reproducible by calling the existing `src/` pipeline directly. The reported model is `xgboost_log`, with metrics and predictions produced by the same validated training and inference code used in the project.
